# Combined Figure: Silhouette Comparison + Replicated Improvement Boxplot

Row 1 is a side-by-side scatter comparison (Original Silhouette vs
rNN-affinity Hard Silhouette **at the selected $r^*$**) on the single
representative dataset from `gaussian_simulation.ipynb`, scored against
ACC; row 2 is the ACC improvement boxplot from
`gaussian_simulation_boxplot.ipynb` (100 reps).

**Important:** `gaussian_simulation.ipynb` never computes $r^*$ -- it
only shows 3 FIXED $r$ values (5, 30, 100), illustratively. There is no
pre-existing $r^*$ for its single representative dataset. This notebook
computes one, using the exact production selection rule
(`select_r_topK_spearman`, $m=35$, $\tau=0.95$) over the wider
$K=2,\ldots,(\widehat K+10)$ window, on the SAME representative dataset
(`random_state=1`, unstandardized, matching `gaussian_simulation.ipynb`
exactly) -- this is new computation, not a reuse of a previously-saved
value. The $r$-grid used for selection matches
`gaussian_simulation_boxplot.ipynb`'s own grid ($5,10,\ldots,100$) so
the two halves of this combined figure stay internally consistent with
each other, rather than importing the production scripts' up-to-130
grid.

The left panel and the boxplot panel are **not** recomputed -- they
reuse the exact cached results from both notebooks' own saved pickles,
so they reproduce byte-identical to the existing standalone figures.

In [ ]:
# -- I/O and core numerical stack --
import os
import pickle

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy.stats import spearmanr

# -- Clustering, synthetic data generation, and silhouette scoring (used
# only in Step 2 below, to recompute r* fresh) --
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

OUT_DIR = 'figures/combined'
os.makedirs(OUT_DIR, exist_ok=True)

# This combined figure is built against ACC specifically (the top row's
# scatter panels and the bottom row's boxplot are both scored vs. ACC).
EXT_METRIC = 'accuracy'
EXT_LABEL = 'ACC'

## Helper functions

(copied verbatim from `gaussian_simulation.ipynb` / `gaussian_simulation_boxplot.ipynb`)

In [ ]:
def rnn_affinity_distance(X, r, use_gaussian=False):
    """Build the rNN affinity distance matrix (Hard if use_gaussian=False,
    Soft/Gaussian-weighted otherwise). Copied verbatim from
    gaussian_simulation.ipynb / gaussian_simulation_boxplot.ipynb -- see
    those notebooks for the fuller explanatory comments. Only the Hard
    variant is actually used in this notebook (for the fresh r* selection
    in Step 2 below); use_gaussian=True is kept only because this helper
    was copied as-is."""
    n = X.shape[0]
    # Query r+1 neighbors and remove self so r counts other observations.
    nbrs = NearestNeighbors(n_neighbors=r + 1, algorithm='auto').fit(X)
    dists, idx = nbrs.kneighbors(X)
    dists, idx = dists[:, 1:], idx[:, 1:]
    A = np.zeros((n, n))
    if use_gaussian:
        for i in range(n):
            sigma = dists[i, -1] or 1e-10
            for j_pos, j in enumerate(idx[i]):
                A[i, j] = np.exp(-dists[i, j_pos]**2 / (2 * sigma**2))
    else:
        for i in range(n):
            A[i, idx[i]] = 1.0
    A = np.maximum(A, A.T)
    np.fill_diagonal(A, 0)
    a_min, a_max = A.min(), A.max()
    if a_max > a_min:
        A = (A - a_min) / (a_max - a_min)
    D = 1.0 - A
    np.fill_diagonal(D, 0)
    return D


def spearmanr_safe(a, b):
    """Spearman correlation, returns NaN on degenerate input (fewer than
    2 valid finite pairs)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() < 2:
        return np.nan
    rho, _ = spearmanr(a[mask], b[mask])
    return float(rho)


def select_r_topK_spearman(sil_dict, r_values, m, n_k, threshold=0.95):
    """Production r* selection rule (Algorithm 1): pick the smallest r
    whose top-m_eff largest-K silhouette values decrease monotonically
    enough (Spearman rho vs. the ideal decreasing sequence >= threshold);
    fall back to the r with the single best such rho if none clears the
    threshold. See gaussian_simulation_boxplot.ipynb for the fuller
    explanatory version of this same function. Unlike that notebook's
    version, this one also returns the full dict of per-r criterion scores
    (c_vals) so Step 2 below can print the winning r*'s score alongside
    the threshold for a sanity check."""
    avail = sorted(r for r in r_values if r in sil_dict)
    if not avail:
        return r_values[0], {}
    m_eff = max(2, min(m, n_k - 1))
    top_idx = list(range(n_k - m_eff, n_k))
    ideal = np.arange(m_eff, 0, -1, dtype=float)
    c_vals = {}
    for r in avail:
        obs = np.asarray(sil_dict[r], dtype=float)[top_idx]
        rho = spearmanr_safe(obs, ideal)
        c_vals[r] = rho if not np.isnan(rho) else -2.0
    r_star = next((r for r in avail if c_vals[r] >= threshold), None)
    if r_star is None:
        r_star = max(avail, key=lambda r: c_vals[r])
    return r_star, c_vals


def simulate_gaussian_data(n_samples=2000, n_features=20, K_star=10,
                            cluster_std=1, random_state=1):
    """Thin wrapper around sklearn's make_blobs -- identical to the
    version in gaussian_simulation.ipynb. Called in Step 2 below with the
    exact same arguments (N=2000, d=20, K_star=10, random_state=1) to
    regenerate that notebook's representative dataset byte-for-byte."""
    X, y = make_blobs(n_samples=n_samples, n_features=n_features,
                       centers=K_star, cluster_std=cluster_std,
                       random_state=random_state)
    return X, y


def max_K_eval(k_star):
    """max(25, 1.75*k*) -- the reference paper's own K-range cap,
    matching the evaluation-scope convention used everywhere else in this
    package."""
    return int(max(25, round(1.75 * k_star)))

## Plotting utilities

In [ ]:
def get_point_colors(K_test, K_star):
    """Blue shades for K < K_star (under-partition), orange shades for
    K > K_star (over-partition); intensity increases the closer K is to
    K_star. Same logic as gaussian_simulation.ipynb's version."""
    K_arr = np.array(K_test)
    dist = np.abs(K_arr - K_star).astype(float)
    max_d = dist.max() if dist.max() > 0 else 1.0
    intensity = 1.0 - dist / max_d
    colors = []
    for i, K in enumerate(K_arr):
        v = 0.3 + 0.7 * intensity[i]
        colors.append(plt.cm.Blues(v) if K < K_star else plt.cm.Oranges(v))
    return colors, intensity


def _add_k_labels(ax, xs, ys, K_vals, inside_fs=10, outside_fs=11,
                   threshold=0.03, offset_frac=0.13, overrides=None):
    """Place K number labels with overlap avoidance (collision detection +
    repulsion-direction offset, with optional manual overrides). See
    gaussian_simulation.ipynb's _add_k_labels for the fully-commented
    version of this same collision/repulsion/override logic -- this copy
    is functionally identical, just with a few font-size defaults tuned
    for this combined figure's larger panels."""
    n = len(xs)
    if n == 0:
        return
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    xspan = max(xlim[1] - xlim[0], 1e-9)
    yspan = max(ylim[1] - ylim[0], 1e-9)
    xn = (xs - xlim[0]) / xspan
    yn = (ys - ylim[0]) / yspan

    # Flag colliding labels (pairwise distance below threshold, in
    # normalized axes-fraction space).
    needs_offset = set()
    for i in range(n):
        for j in range(i + 1, n):
            if np.hypot(xn[i] - xn[j], yn[i] - yn[j]) < threshold:
                needs_offset.add(i)
                needs_offset.add(j)

    # Repulsion direction: sum of unit vectors away from nearby points.
    offset_dir = {}
    for i in sorted(needs_offset):
        fx, fy = 0.0, 0.0
        for j in range(n):
            if j == i:
                continue
            dx, dy = xn[i] - xn[j], yn[i] - yn[j]
            d = np.hypot(dx, dy)
            if 0 < d < threshold * 5:
                fx += dx / d
                fy += dy / d
        norm = np.hypot(fx, fy)
        fx, fy = (fx / norm, fy / norm) if norm > 1e-6 else ((0.0, 1.0) if i % 2 == 0 else (0.0, -1.0))
        offset_dir[i] = (fx, fy)

    # Manual overrides (per-K: None = force inside, (fx,fy) = force outside
    # in that direction) take priority over the automatic decision.
    if overrides:
        for i in range(n):
            k = int(K_vals[i])
            if k not in overrides:
                continue
            direction = overrides[k]
            if direction is None:
                needs_offset.discard(i)
                offset_dir.pop(i, None)
            else:
                needs_offset.add(i)
                offset_dir[i] = direction

    for i in range(n):
        x, y, k = float(xs[i]), float(ys[i]), int(K_vals[i])
        if i not in needs_offset:
            ax.text(x, y, str(k), ha='center', va='center',
                    fontsize=inside_fs, fontweight='bold', color='white', zorder=6)
        else:
            fx, fy = offset_dir[i]
            tx = x + fx * offset_frac * xspan
            ty = y + fy * offset_frac * yspan
            ax.annotate(str(k), xy=(x, y), xytext=(tx, ty),
                        ha='center', va='center', fontsize=outside_fs,
                        fontweight='bold', color='black',
                        arrowprops=dict(arrowstyle='-', color='gray', lw=0.6, shrinkA=0, shrinkB=4),
                        zorder=6)


def plot_panel(ax, sil, ext, K_test, K_star, title, ext_label, x_label,
               title_fs=20, subtitle_fs=18, x_label_fs=16, y_label_fs=10,
               tick_fs=9, label_overrides=None):
    """Draw one scatter panel (silhouette-type score vs. external metric),
    with the Spearman rho shown as a subtitle line below the panel title
    (rather than inside the title string, as gaussian_simulation.ipynb
    does) -- this figure's title is set directly by the caller (Step 3
    below: 'Ordinary Silhouette Score' / 'rNN-based Silhouette Score'), so
    rho is placed separately via ax.text instead of being interpolated
    into it. Returns rho so the caller could use/print it if desired
    (unused by the current Step 3 code, which recomputes rho implicitly
    through this call but doesn't capture the return value)."""
    K_arr = np.array(K_test)
    colors, intensity = get_point_colors(K_test, K_star)
    valid_all = ~np.isnan(sil) & ~np.isnan(ext)

    for mask, cmap_fn, marker, ms in [
        (K_arr < K_star, plt.cm.Blues, 'o', 200),
        (K_arr > K_star, plt.cm.Oranges, 's', 260),
    ]:
        valid = mask & valid_all
        if not np.any(valid):
            continue
        c_valid = [colors[i] for i in np.where(valid)[0]]
        ax.plot(sil[valid], ext[valid], lw=1.5, color=cmap_fn(0.5), alpha=0.3, zorder=1)
        ax.scatter(sil[valid], ext[valid], c=c_valid, s=ms, marker=marker,
                   zorder=3, edgecolors='white', linewidths=0.4)

    if valid_all.any():
        best_i = int(np.nanargmax(ext))
        ax.scatter(sil[best_i], ext[best_i], marker='*', s=350, color='red',
                   zorder=5, edgecolors='darkred', linewidths=0.5)

    _add_k_labels(ax, sil[valid_all], ext[valid_all], K_arr[valid_all], overrides=label_overrides)

    rho = spearmanr(sil[valid_all], ext[valid_all])[0] if valid_all.sum() >= 2 else np.nan
    best_val = np.nanmax(ext)
    best_K = K_arr[int(np.nanargmax(ext))]

    ax.set_title(title, fontsize=title_fs, fontweight='bold', pad=40)
    ax.text(0.5, 1.01,
            f'Spearman $\\rho$: {rho:.3f}',
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=subtitle_fs, fontweight='bold')
    ax.set_xlabel(x_label, fontsize=x_label_fs)
    ax.set_ylabel(ext_label, fontsize=y_label_fs)
    ax.tick_params(labelsize=tick_fs)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.grid(True, linestyle=':', alpha=0.4)
    return rho


def build_legend_handles():
    """Proxy legend entries (best-K star / under-partition circle /
    over-partition square) for the top row's shared legend."""
    from matplotlib.lines import Line2D
    return [
        Line2D([0], [0], marker='*', color='none', markerfacecolor='red',
               markeredgecolor='darkred', markersize=15, label='Best $K$'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='steelblue',
               markeredgecolor='white', markersize=12, label='$K <$ True $K^*$  (under-partition)'),
        Line2D([0], [0], marker='s', color='none', markerfacecolor='darkorange',
               markeredgecolor='white', markersize=13, label='$K >$ True $K^*$  (over-partition)'),
    ]

## Boxplot helper

In [ ]:
def plot_one_panel(ax, all_reps_diff, r_values, ref_key, ext_label):
    """Draw the bottom-row boxplot panel: for each r in r_values plus a
    final r* group, a pair of boxplots (Hard/Soft) of the per-rep
    Spearman-rho improvement over the ordinary silhouette. Adapted from
    gaussian_simulation_boxplot.ipynb's plot_one_panel -- identical
    logic, minus that version's own ax.set_title call (Step 3 below sets
    this panel's title itself, so it isn't duplicated here)."""
    variants = ['hard', 'soft']
    colors = {'hard': 'steelblue', 'soft': 'darkorange'}
    labels_legend = {
        'hard': 'Silhouette Score (Neighborhood Hard)',
        'soft': 'Silhouette Score (Neighborhood Soft)',
    }
    width = 0.28
    offsets_rel = [-width / 2, width / 2]
    plot_keys = list(r_values) + ['r_star']
    plot_labels = [f'r = {r}' for r in r_values] + [r'$r^*$']
    x_ticks, x_tick_labels, proxy_patches = [], [], {}

    for i, (rkey, rlabel) in enumerate(zip(plot_keys, plot_labels)):
        center = i * (2 * width + 0.5)
        is_r_star = rkey == 'r_star'
        for variant, offset in zip(variants, offsets_rel):
            data = all_reps_diff[rkey][ref_key][variant]
            pos = center + offset
            bp = ax.boxplot([data], positions=[pos], widths=width * 0.9,
                            patch_artist=True, medianprops=dict(color='black', linewidth=2),
                            manage_ticks=False)
            patch = bp['boxes'][0]
            patch.set_facecolor(colors[variant])
            patch.set_alpha(0.85 if is_r_star else 0.65)
            if is_r_star:
                patch.set_hatch('///')
            if variant not in proxy_patches:
                proxy_patches[variant] = patch
        x_ticks.append(center)
        x_tick_labels.append(rlabel)

    # Vertical separator + red-edged highlight box marking the r* group,
    # so it reads visually distinct from the illustrative fixed-r groups.
    sep_x = (len(r_values) - 1) * (2 * width + 0.5) + (2 * width + 0.5) * 0.75
    ax.axvline(sep_x, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)
    pad = width + 0.3
    left_x = x_ticks[len(r_values)] - pad
    right_x = x_ticks[-1] + pad
    ax.axvspan(left_x, right_x, facecolor='none', edgecolor='red', linewidth=2.5, zorder=10)
    ax.axhline(0, color='grey', linewidth=1, linestyle='--')
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels, fontsize=16)
    ax.tick_params(axis='y', labelsize=16)
    ax.set_ylabel(f'Difference in Spearman $\\rho$ with {ext_label}', fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')
    return proxy_patches, labels_legend

## Step 1 — reproduce the left panel's data exactly (cached)

In [ ]:
# Reload gaussian_simulation.ipynb's cached K-sweep results exactly as it
# saved them -- this reproduces the left ("Ordinary Silhouette Score")
# panel's data byte-identically without recomputing anything.
with open('data/gaussian_simulation_results_paperkmax_buffer10_nonself.pkl', 'rb') as f:
    cached = pickle.load(f)
results = cached['results']
K_TEST_EVAL = cached['K_TEST']       # 2..25
K_STAR = cached['K_STAR']            # 10
K_MAX = cached['K_MAX']              # 25
N, d = cached['N'], cached['d']

sil_original = results['sil_original']
ext_vals = results[EXT_METRIC]

print(f'Loaded cached results: K_STAR={K_STAR}, K_MAX={K_MAX}, N={N}, d={d}')

## Step 2 — compute $r^*$ fresh for this same representative dataset

`gaussian_simulation.ipynb` never selects $r^*$ -- this is new
computation, using the exact same data (`random_state=1`,
unstandardized) and the exact production selection rule, over the wider
$K=2,\ldots,(\widehat K+10)$ window.

In [ ]:
# Step 2: gaussian_simulation.ipynb never computes r* for its
# representative dataset (it only shows 3 fixed r values). To draw the
# right ("rNN-based Silhouette Score") panel at r*, this cell regenerates
# that exact same dataset (random_state=1, unstandardized) and runs the
# production r*-selection rule (select_r_topK_spearman, m=35, tau=0.95)
# fresh on it.
print('Regenerating the exact representative dataset (random_state=1)...')
X, y_true = simulate_gaussian_data(n_samples=N, n_features=d, K_star=K_STAR,
                                    cluster_std=1, random_state=1)

# r*-selection r-grid: matches gaussian_simulation_boxplot.ipynb's own grid
# (5, 10, ..., 130) rather than the production scripts' grid, so this
# combined figure's two halves stay internally consistent with each other.
R_VALUES_SELECT = list(range(5, 135, 5))   # matches the production scripts' grid (Part A/B/Benchmark_data_real)
M, THRESHOLD = 35, 0.95
# Selection uses the WIDER K-range (evaluation cap + buffer b=10), per
# Algorithm 1 -- distinct from K_TEST_EVAL (2..25) loaded from the cache
# above, which is only the narrower range used for reporting/plotting.
K_MAX_SELECT = max_K_eval(K_STAR) + 10     # buffer b=10, matching Algorithm 1
K_TEST_SELECT = list(range(2, K_MAX_SELECT + 1))   # 2..35
print(f'K_TEST_EVAL=2..{K_MAX} ({len(K_TEST_EVAL)} values, from cache)')
print(f'K_TEST_SELECT=2..{K_MAX_SELECT} ({len(K_TEST_SELECT)} values, computing fresh)')

# Cluster K=2..35 with the same random_state=42 gaussian_simulation.ipynb
# used for its own K-sweep, so the labels (and hence the silhouette curves
# used for selection) match that notebook's clustering exactly.
print('Clustering K=2..35 (random_state=42, matching gaussian_simulation.ipynb)...')
labels_by_K = {K: KMeans(n_clusters=K, random_state=42, n_init='auto').fit_predict(X)
               for K in K_TEST_SELECT}

# Build the rNN-Hard silhouette curve (over the WIDE K_TEST_SELECT range)
# for every candidate r on the selection grid; select_r_topK_spearman below
# then picks r* from these curves' shapes alone (external metrics are
# never consulted during selection).
print(f'Computing rNN-Hard silhouette for r in {R_VALUES_SELECT}...')
sil_hard_by_r = {}
for r in R_VALUES_SELECT:
    D_hard = rnn_affinity_distance(X, r=r, use_gaussian=False)
    sil_hard_by_r[r] = np.array([
        silhouette_score(D_hard, labels_by_K[K], metric='precomputed')
        if len(np.unique(labels_by_K[K])) > 1 else np.nan
        for K in K_TEST_SELECT
    ])

r_star, c_vals = select_r_topK_spearman(sil_hard_by_r, R_VALUES_SELECT, m=M,
                                          n_k=len(K_TEST_SELECT), threshold=THRESHOLD)
print(f'Selected r* = {r_star}  (C(r*)={c_vals[r_star]:.4f}, threshold={THRESHOLD})')

# Slice the r*-selected curve down to the EVAL range (K=2..25) for plotting,
# matching the left panel's K range exactly.
eval_idx_in_select = [K_TEST_SELECT.index(K) for K in K_TEST_EVAL]
sil_hard_rstar_eval = sil_hard_by_r[r_star][eval_idx_in_select]

## Step 3 — build the combined figure

In [ ]:
# Step 3: assemble the final 2-row figure.
#   Row 1 (top):    two side-by-side scatter panels -- ordinary Silhouette
#                    (left, from the cached K-sweep) vs. rNN-based
#                    Silhouette at r* (right, freshly computed in Step 2) --
#                    both scored against ACC on the representative dataset.
#   Row 2 (bottom):  the 100-replicate ACC improvement boxplot, reusing
#                    gaussian_simulation_boxplot.ipynb's cached results.
# Panel-specific `label_overrides` below are hand-tuned K-label placements
# (see _add_k_labels' docstring above) fixing collisions the automatic
# repulsion heuristic didn't fully resolve for this particular figure
# layout.
fig = plt.figure(figsize=(17, 14))
outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 1],
                          hspace=0.72, left=0.07, right=0.97,
                          bottom=0.20, top=0.93)
top_grid = outer[0].subgridspec(1, 2, wspace=0.12)
bottom_grid = outer[1].subgridspec(1, 3, width_ratios=[0.3, 1, 0.3],
                                      wspace=0)

ax_left = fig.add_subplot(top_grid[0, 0])
plot_panel(ax_left, sil_original, ext_vals, K_TEST_EVAL, K_STAR,
           title='Ordinary Silhouette Score', ext_label=EXT_LABEL,
           x_label='Index Values',
           title_fs=20, subtitle_fs=18, x_label_fs=20, y_label_fs=20, tick_fs=10,
           label_overrides={K_TEST_EVAL[-1]: None, K_STAR: None, 14: None,
                            20: None, 21: (1.0, 1.0)})

ax_right = fig.add_subplot(top_grid[0, 1])
plot_panel(ax_right, sil_hard_rstar_eval, ext_vals, K_TEST_EVAL, K_STAR,
           title='rNN-based Silhouette Score', ext_label=EXT_LABEL,
           x_label='Index Values',
           title_fs=20, subtitle_fs=18, x_label_fs=20, y_label_fs=20, tick_fs=10,
           label_overrides={K_TEST_EVAL[-1]: None, K_STAR: None,
                            12: None, 15: None, 17: None,
                            19: (0.8, -0.7), 20: None})

# -- Bottom panel: the ACC improvement boxplot (100 reps, cached) --
with open('data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl', 'rb') as f:
    saved = pickle.load(f)
all_reps_diff = saved['all_reps_diff']
R_VALUES_PLOT = saved.get('R_VALUES_PLOT', [5, 10, 30, 50, 100])
N_REPS = saved['N_REPS']

ax_box = fig.add_subplot(bottom_grid[0, 1])
proxy_patches, labels_legend = plot_one_panel(ax_box, all_reps_diff, R_VALUES_PLOT, 'acc', EXT_LABEL)
ax_box.set_title('Improvement of $r$NN-based Silhouette Score over the Ordinary one',
                 fontsize=20, fontweight='bold', pad=24)

fig.canvas.draw()

# -- legends below their corresponding panels --
# Legends are positioned using each panel's actual rendered bounding box
# (via get_position(), computed only after fig.canvas.draw() above lays
# out the figure) rather than fixed coordinates, so the legend placement
# adapts correctly regardless of exact panel sizing.
pos_left = ax_left.get_position()
pos_right = ax_right.get_position()
top_legend = fig.legend(handles=build_legend_handles(), loc='upper center', ncol=3,
                        fontsize=20, framealpha=0.9, columnspacing=0.9, handletextpad=0.5,
                        bbox_to_anchor=((pos_left.x0 + pos_right.x1) / 2,
                                        pos_left.y0 - 0.055))

handles = [proxy_patches[v] for v in ('hard', 'soft')]
labels = [labels_legend[v] for v in ('hard', 'soft')]
pos_box = ax_box.get_position()
bottom_legend = fig.legend(handles, labels, fontsize=20, loc='upper center', ncol=2,
                           framealpha=0.9, columnspacing=0.9, handletextpad=0.5,
                           bbox_to_anchor=((pos_box.x0 + pos_box.x1) / 2,
                                           pos_box.y0 - 0.075))

# -- panel labels immediately left of their legends, on the same line --
# '(a)'/'(b)' panel labels are placed relative to each legend's own
# rendered bounding box (re-queried via get_window_extent after another
# canvas draw, since the legends were only just added above), so they stay
# aligned with the legend row even if legend content/size changes.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
top_bbox = top_legend.get_window_extent(renderer=renderer).transformed(fig.transFigure.inverted())
bottom_bbox = bottom_legend.get_window_extent(renderer=renderer).transformed(fig.transFigure.inverted())
fig.text(top_bbox.x0 - 0.015, (top_bbox.y0 + top_bbox.y1) / 2, '(a)',
         ha='right', va='center', fontsize=18, fontweight='bold')
fig.text(bottom_bbox.x0 - 0.015, (bottom_bbox.y0 + bottom_bbox.y1) / 2, '(b)',
         ha='right', va='center', fontsize=18, fontweight='bold')

# This is the figure referenced in main.tex's "Mechanism-Focused Gaussian
# Simulation" section.
out_path = os.path.join(OUT_DIR, 'gaussian_combined_landscape_boxplot.pdf')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'\nSaved: {out_path}')
plt.show()